# Lab 7: Input Modelling, Validation, and a Final Recommendation

Complete the code TODOs, the written answers (every **ANSWER:** box), the manager recommendation, the reflection, and the AI-use statement. Every code location that you must complete is marked `TODO`; code marked **Provided** or **do not edit** should be left unchanged. This notebook is both the practical work and the report.

## Student Information

**Name:** Write your answer here

**Student ID:** Write your answer here

**Date:** Write your answer here

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Provided lab settings -- do not edit.
SEED = 211               # seed for the fitted sample in Part B; replication r uses seed SEED + r
DAY_LENGTH = 480         # minutes the shop accepts arrivals (8 hours)
ARRIVAL_RATE = 12        # customers per hour
N_REPLICATIONS = 30      # days in Parts D and E


def locate_data():
    # Provided: find data/service_times.csv from the repository root or from a lab folder.
    candidates = ["data/service_times.csv", "../../data/service_times.csv", "../../../data/service_times.csv"]
    for candidate in candidates:
        if os.path.exists(candidate):        # does this file exist?
            return candidate
    raise FileNotFoundError("Run from the repository root or from the lab folder.")

## Part A: Check the Observed Input Data

Chapter 9 starts input modelling with the raw data, **before** fitting anything. The file `data/service_times.csv` contains 20 observed service times in minutes.

In [ ]:
observed = TODO

summary = pd.Series({
    "count": TODO,
    "mean": TODO,
    "sample SD": TODO,
    "minimum": TODO,
    "median": TODO,
    "maximum": TODO,
})
summary["coefficient of variation"] = TODO

# Provided raw-data checks -- do not edit.
assert len(observed) == 20, "Use data/service_times.csv (20 rows), not an older local copy."
assert (observed > 0).all(), "Service times must be positive."
summary.round(3)

In [ ]:
# Provided histogram -- do not edit.
plt.hist(observed, bins=np.arange(0, 4.01, 0.5), edgecolor="black")
plt.xlabel("Service time (minutes)")
plt.ylabel("Count")
plt.title("20 observed service times")
plt.show()

### Part A Questions

1. Describe a typical service time, the spread, and the shape of the histogram.

   **ANSWER:** Write your answer here.

2. An exponential distribution always has coefficient of variation (SD / mean) equal to 1. What does the observed value suggest before any fitting?

   **ANSWER:** Write your answer here.

## Part B: Fit an Exponential Distribution and Check It

For an exponential distribution $\mathbb E[S]=1/\lambda$, and the maximum-likelihood estimate is $\hat\lambda=1/\bar x$. 

Fitting **matches the mean by construction**; the question is whether the **shape** is credible.

In [ ]:
mean_service = TODO          # fitted exponential mean (minutes)
rate = TODO                 # fitted rate (per minute)
fitted_sample = TODO

print(f"fitted mean = {mean_service:.3f} min,  rate = {rate:.4f} per min")

In [ ]:
# Provided graphical checks -- do not edit.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(observed, bins=np.arange(0, 8.01, 0.5), density=True, alpha=0.6, edgecolor="black", label="observed (20)")
axes[0].hist(fitted_sample, bins=np.arange(0, 8.01, 0.5), density=True, alpha=0.4, label="fitted exponential")
axes[0].set(xlabel="Service time (minutes)", ylabel="Density", title="Histogram comparison")
axes[0].legend()
axes[1].step(np.sort(observed), np.arange(1, 21) / 20, where="post", label="observed ECDF")
grid = np.linspace(0, 8, 200)
axes[1].plot(grid, 1 - np.exp(-grid / mean_service), label="fitted exponential CDF")
axes[1].set(xlabel="Service time (minutes)", ylabel="Cumulative probability", title="ECDF comparison")
axes[1].legend()
plt.tight_layout()
plt.show()

### Chi-square goodness-of-fit test

With only 20 observations, use $k=4$ bins that each have probability $\tfrac14$ under the fitted exponential, so every expected count is $E_i=20/4=5$ (Chapter 9 asks for $E_i\ge5$). The bin edges are the fitted quartiles: solving $1-e^{-x/\bar x}=q$ gives

$$
x_q=-\bar x\,\ln(1-q),\qquad q=\tfrac14,\ \tfrac12,\ \tfrac34 .
$$

One parameter was estimated from the same data, so $\mathrm{df}=k-1-1=2$, and the 5% critical value is $\chi^2_{0.05,2}=5.991$.

In [ ]:
inner_edges = TODO
edges = np.concatenate([[0], inner_edges, [np.inf]])       # Provided
observed_counts, _ = np.histogram(observed, bins=edges)    # Provided
expected_counts = np.full(4, len(observed) / 4)            # Provided

chi_square = TODO

# Provided table -- do not edit.
print(f"chi-square = {chi_square:.2f}   (5% critical value with df = 2: 5.991)")
pd.DataFrame({"lower": edges[:-1], "upper": edges[1:], "observed": observed_counts,
              "expected": expected_counts}).round(3)

### Part B Questions

1. Judge the exponential fit. Use the histogram, the ECDF, the coefficient of variation, and the chi-square result. Where exactly does it fit poorly?

   **ANSWER:** Write your answer here.

2. Why does matching the sample mean **not** validate the exponential model?

   **ANSWER:** Write your answer here.

3. With only 20 observations and 4 bins, how much weight should one test result carry?

   **ANSWER:** Write your answer here.

## Part C: A One-Day Model

The model for one day:

- customers arrive with exponential gaps at `ARRIVAL_RATE` per hour; arrivals are accepted until minute `DAY_LENGTH`, and everyone who arrived is served;
- one barista, first come first served, no customer leaves the line;
- service times come from a **service model**: `"exponential"` (the Part B fit) or `"empirical"` (resample the 20 observed values with replacement, Chapter 9's non-parametric approach).

All arrival times are generated first and all service times second, so two service models run with the same seed see **the same customers arriving at the same times** (Lab 5).

In [ ]:
def simulate_day(service_model, seed):
    rng = np.random.default_rng(seed)

    # Arrival times: keep adding exponential gaps while the shop still accepts arrivals.
    arrivals = []
    t = 0.0
    t = t + TODO                      # the first arrival: time 0 plus one gap
    while t <= DAY_LENGTH:
        arrivals.append(t)
        t = t + TODO                  # the next arrival: now plus one gap (the same as above)
    arrival = np.array(arrivals)
    n = len(arrival)

    # Service times from the chosen model.
    if service_model == "exponential":
        service = TODO
    elif service_model == "empirical":
        service = rng.choice(observed, size=n, replace=True)      # Provided: resample the data
    else:
        raise ValueError("service_model must be 'exponential' or 'empirical'")

    # FCFS recursion (Lab 5).
    start = np.empty(n)
    end = np.empty(n)
    previous_end = 0.0                     # the shop opens empty
    for i in range(n):
        start[i] = TODO
        end[i] = TODO
        previous_end = end[i]              # this customer's end time, for the next customer

    # Provided output measures -- do not edit.
    wait = start - arrival
    in_line = []                           # customers already waiting in line when customer i arrives
    for i in range(n):
        in_line.append(np.sum(start[:i] > arrival[i]))
    return {
        "average_wait": wait.mean(),
        "max_wait": wait.max(),
        "max_queue_length": int(max(in_line)),
        "number_served": n,
    }


one_day = simulate_day("exponential", SEED + 1)
pd.Series(one_day).round(3)

### Part C Question

1. This one day (seed 212) had an average wait of about 3.7 minutes. Is that what a typical day looks like? How would you find out?

   **ANSWER:** Write your answer here.

## Part D: Replications, Verification, and a Stopping Rule

Chapter 9: each complete day is **one replication**, giving one value of each output. Independent days give independent values, so the Lab 4 interval applies:

$$
\bar X\pm1.96\,\frac{s}{\sqrt n}.
$$

Replication $r=1,\dots,30$ uses seed `SEED + r`.

In [ ]:
results = []
for r in range(1, N_REPLICATIONS + 1):
    results.append(TODO)          # one day: simulate_day with the exponential model and seed SEED + r
exponential_days = pd.DataFrame(results)     # Provided: one row per day, one column per output


def confidence_interval(values, z=1.96):
    values = np.asarray(values, dtype=float)
    mean = TODO
    sd = TODO
    half_width = TODO
    return mean, sd, mean - half_width, mean + half_width


# Provided table -- do not edit.
measures = ["average_wait", "max_wait", "max_queue_length", "number_served"]
rows = []
for m in measures:
    mean, sd, lower, upper = confidence_interval(exponential_days[m])
    rows.append({"measure": m, "mean": mean, "sd": sd, "lower": lower, "upper": upper})
ci_exponential = pd.DataFrame(rows).set_index("measure")
ci_exponential.round(3)

In [ ]:
# Provided output display -- do not edit.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(exponential_days["average_wait"], bins=10, edgecolor="black")
axes[0].set(xlabel="daily average wait (min)", ylabel="days", title="30 days: daily average wait")
axes[1].hist(exponential_days["max_wait"], bins=10, edgecolor="black")
axes[1].set(xlabel="longest wait of the day (min)", ylabel="days", title="30 days: longest wait")
plt.tight_layout()
plt.show()

### Verification against an exact benchmark

Verification asks whether the **code** implements the intended model. With exponential service this model is an M/M/1 queue with $\lambda=12$ per hour $=0.2$ per minute and $\mu=1/\bar x$ per minute. Its steady-state mean wait in the queue is

$$
W_q=\frac{\rho}{\mu-\lambda},\qquad \rho=\frac{\lambda}{\mu}\quad\text{(Chapter 7)}.
$$

An 8-hour day is a terminating simulation that starts empty, but at this low load the start-up period is short, so the replication mean should be close to $W_q$. A large disagreement would point to a bug first.

In [ ]:
lam = ARRIVAL_RATE / 60                 # arrivals per minute
mu = 1 / mean_service                   # services per minute
rho = TODO
W_q = TODO                # minutes

# Provided comparison -- do not edit.
mean_w, sd_w, lower_w, upper_w = confidence_interval(exponential_days["average_wait"])
print(f"rho = {rho:.3f},  M/M/1 W_q = {W_q:.3f} min")
print(f"simulated mean of daily average waits = {mean_w:.3f} min,  95% CI [{lower_w:.3f}, {upper_w:.3f}]")
print("W_q inside the interval?", lower_w <= W_q <= upper_w)

### A precision-driven stopping rule

How many days are needed for a 95% margin of error below $d=0.1$ minutes for the average wait? Using the sample SD from the 30 days,

$$
k\ \ge\ \left(\frac{1.96\,s}{d}\right)^2 .
$$

In [ ]:
d = 0.1
k_needed = TODO

# Provided check: run k_needed days and report the achieved margin -- do not edit.
more_waits = []
for r in range(1, k_needed + 1):
    more_waits.append(simulate_day("exponential", SEED + r)["average_wait"])
mean_k, sd_k, lower_k, upper_k = confidence_interval(more_waits)
print(f"k needed = {k_needed}")
print(f"with {k_needed} days: mean = {mean_k:.3f}, 95% CI [{lower_k:.3f}, {upper_k:.3f}], margin = {(upper_k - lower_k) / 2:.3f}")

### Part D Questions

1. Which output is the most uncertain relative to its mean? What does the interval for the average wait say that the single day in Part C could not?

   **ANSWER:** Write your answer here.

2. Does the model pass the verification check? Explain what a pass does and does not show.

   **ANSWER:** Write your answer here.

3. If the exponential input model is wrong, will running the stopping-rule number of days fix that?

   **ANSWER:** Write your answer here.

## Part E: Sensitivity to the Input Model (Validation)

Part B suggested the exponential shape is doubtful. **Sensitivity analysis** is a validation check: do the conclusions change when an uncertain assumption is varied? Rerun the same 30 days with the empirical service model. The same seeds give the same arrivals, so the comparison is paired.

In [ ]:
results = []
for r in range(1, N_REPLICATIONS + 1):
    results.append(TODO)                  # one day: simulate_day with the empirical model and seed SEED + r
empirical_days = pd.DataFrame(results)    # Provided: one row per day, one column per output

# Provided comparison -- do not edit.
rows = []
for m in ["average_wait", "max_wait", "max_queue_length"]:
    for name, days in [("exponential", exponential_days), ("empirical", empirical_days)]:
        mean, sd, lower, upper = confidence_interval(days[m])
        rows.append({"measure": m, "service model": name, "mean": mean, "lower": lower, "upper": upper})
difference = exponential_days["average_wait"] - empirical_days["average_wait"]
mean_d, sd_d, lower_d, upper_d = confidence_interval(difference)
print(f"paired difference in average wait (exponential - empirical): {mean_d:.3f} min, 95% CI [{lower_d:.3f}, {upper_d:.3f}]")
print(f"same number of customers each day? {np.array_equal(exponential_days['number_served'], empirical_days['number_served'])}")
pd.DataFrame(rows).set_index(["measure", "service model"]).round(3)

### Part E Questions

1. How much does the choice of input model change the average wait and the longest wait? Which model would you trust more for this shop, and what is the main weakness of each?

   **ANSWER:** Write your answer here.

2. Is the model credible enough to support a staffing recommendation? Name the evidence for and against.

   **ANSWER:** Write your answer here.

3. Why does the input model change the average wait so much when both models have the same mean service time?

   **ANSWER:** Write your answer here.

## Recommendation, Reflection, and AI Use

**Manager recommendation:** The manager asks whether one barista is enough for a typical day. Give a specific decision, numerical results with confidence intervals, a comment on the service-time model (fit and sensitivity), one limitation that could change the conclusion, and one practical next action. Write your answer here.

**Reflection:** What is the difference between verification and validation, and what evidence of each did you produce? Why are replications better than one day? What did the sensitivity analysis add? Write your answer here.

**AI-use statement:** If AI was used, identify the tool, how it was used, and how you checked or corrected its output. If AI was not used, say so. You must be able to explain all submitted work. Write your answer here.

## Final Submission

Complete Parts A-E, the recommendation, the reflection, and the AI-use statement. Then:

1. save the notebook as `Lab07_StudentID.ipynb`, for example `Lab07_6900789.ipynb`;
2. restart the kernel and run all cells from top to bottom;
3. check that there are no errors;
4. save the notebook; and
5. upload the single completed `.ipynb` file to the **Lab 07** assignment in Google Classroom.